# Decision Tree for Heart Failure Prediction
> **Dataset:** Heart Failure Clinical Records (UCI, 299 patients, 12 features)  
> **Target:** `DEATH_EVENT` (binary: 0 = survived, 1 = died)  
> **Author:** Pranav Ananth  
> **Reference:** Chicco & Jurman (2020), BMC Medical Informatics

---
## Algorithm Overview
**Decision Trees** recursively partition the feature space using axis-aligned splits. At each node, the split is chosen to maximize information gain (or minimize Gini impurity):

$$\text{Gini} = 1 - \sum_k p_k^2 \qquad \text{Entropy} = -\sum_k p_k \log_2 p_k$$

**Strengths for this project:**  
- Fully interpretable (can be visualized)  
- No feature scaling required  
- Captures non-linear relationships and interactions  
- Directly outputs feature importances

## 1 · Setup & Imports

In [ ]:
# Install required packages
!pip install -q pandas numpy scikit-learn matplotlib seaborn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    matthews_corrcoef, roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, confusion_matrix, roc_curve,
    precision_recall_curve, classification_report, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ All packages imported successfully")

## 2 · Load & Explore Data

In [ ]:
# ── Option A: upload file manually ──────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_excel(list(uploaded.keys())[0])

# ── Option B: load directly from UCI ────────────────────────────────────────
import urllib.request
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00519/heart_failure_clinical_records_dataset.csv"
urllib.request.urlretrieve(url, "heart_failure.csv")
df = pd.read_csv("heart_failure.csv")

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Quick EDA
print("=== Class Distribution ===")
print(df['DEATH_EVENT'].value_counts())
print(f"\nClass imbalance ratio: {df['DEATH_EVENT'].value_counts()[0]/df['DEATH_EVENT'].value_counts()[1]:.2f}:1")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
df['DEATH_EVENT'].value_counts().plot(kind='bar', ax=axes[0], color=['#3266ad','#e05a4a'], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_xticklabels(['Survived (0)', 'Died (1)'], rotation=0)
axes[0].set_ylabel('Count')

# Feature correlation heatmap
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=axes[1], cmap='RdBu_r', center=0,
            linewidths=0.5, annot=False, fmt='.2f')
axes[1].set_title('Feature Correlation Matrix', fontsize=13)

plt.tight_layout()
plt.show()

## 3 · Train / Test Split

In [ ]:
# Feature / target split
feature_cols = [c for c in df.columns if c != 'DEATH_EVENT']
X = df[feature_cols]
y = df['DEATH_EVENT']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test positive rate: {y_test.mean():.3f}")

## 4 · Evaluation Utilities

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "MCC":       round(matthews_corrcoef(y_test, y_pred), 4),
        "AUC-ROC":   round(roc_auc_score(y_test, y_proba), 4),
        "F1":        round(f1_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
    }

    print(f"\n{'='*50}")
    print(f"  {model_name} — Test Set Results")
    print(f"{'='*50}")
    for k, v in metrics.items():
        print(f"  {k:<12}: {v}")
    print()
    print(classification_report(y_test, y_pred, target_names=['Survived','Died']))
    return metrics, y_pred, y_proba

def plot_results(model, X_test, y_test, y_pred, y_proba, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'{model_name} — Evaluation Plots', fontsize=14, fontweight='bold')

    # Confusion matrix
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                           display_labels=['Survived','Died']).plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Confusion Matrix')

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[1].plot(fpr, tpr, color='#3266ad', lw=2, label=f'AUC = {auc:.3f}')
    axes[1].plot([0,1],[0,1],'--', color='gray', lw=1)
    axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve'); axes[1].legend()

    # Precision-Recall curve
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    axes[2].plot(rec, prec, color='#e05a4a', lw=2)
    axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
    axes[2].set_title('Precision-Recall Curve')
    axes[2].axhline(y=y_test.mean(), linestyle='--', color='gray', lw=1, label='Baseline')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

def plot_learning_curve(model, X, y, model_name):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        train_sizes=np.linspace(0.1, 1.0, 8), scoring='roc_auc', n_jobs=-1
    )
    plt.figure(figsize=(8, 4))
    plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#3266ad', label='Train AUC')
    plt.fill_between(train_sizes,
                     train_scores.mean(1)-train_scores.std(1),
                     train_scores.mean(1)+train_scores.std(1), alpha=0.1, color='#3266ad')
    plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#e05a4a', label='CV Val AUC')
    plt.fill_between(train_sizes,
                     val_scores.mean(1)-val_scores.std(1),
                     val_scores.mean(1)+val_scores.std(1), alpha=0.1, color='#e05a4a')
    plt.xlabel('Training Set Size'); plt.ylabel('AUC-ROC')
    plt.title(f'{model_name} — Learning Curve'); plt.legend()
    plt.tight_layout(); plt.show()

print("✅ Evaluation utilities ready")

In [ ]:
def cross_validate_model(model, X, y, model_name):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scorers = ['matthews_corrcoef','roc_auc','f1','precision','recall','accuracy']
    print(f"\n5-Fold Cross-Validation — {model_name}")
    print("-" * 40)
    for s in scorers:
        scores = cross_val_score(model, X, y, cv=cv, scoring=s)
        print(f"  {s:<22}: {scores.mean():.4f} ± {scores.std():.4f}")

print("✅ Cross-validation utility ready")

## 5 · Model Training

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

dt_model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,          # shallow tree → better generalization
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

metrics, y_pred, y_proba = evaluate_model(
    dt_model, X_train, X_test, y_train, y_test, "Decision Tree"
)

## 6 · Evaluation Plots

In [ ]:
plot_results(dt_model, X_test, y_test, y_pred, y_proba, "Decision Tree")

## 7 · Cross-Validation

In [ ]:
cross_validate_model(
    DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42),
    X, y, "Decision Tree"
)

## 8 · Tree Visualization

In [ ]:
dt_model.fit(X_train, y_train)
plt.figure(figsize=(20, 8))
plot_tree(dt_model, feature_names=feature_cols,
          class_names=['Survived','Died'],
          filled=True, rounded=True, fontsize=9,
          impurity=True, proportion=False)
plt.title('Decision Tree (max_depth=4)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(export_text(dt_model, feature_names=feature_cols))

## 9 · Feature Importance

In [ ]:
importances = pd.Series(dt_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
colors = ['#3266ad' if v > 0.05 else '#a0b8d8' for v in importances]
importances.plot(kind='barh', color=colors, edgecolor='white')
plt.title('Decision Tree Feature Importances', fontsize=13)
plt.xlabel('Gini importance')
plt.tight_layout(); plt.show()

## 10 · Max Depth Sweep

In [ ]:
depths = range(1, 15)
train_aucs, val_aucs = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    val  = cross_val_score(m, X, y, cv=5, scoring='roc_auc').mean()
    m.fit(X_train, y_train)
    tr = roc_auc_score(y_train, m.predict_proba(X_train)[:,1])
    train_aucs.append(tr); val_aucs.append(val)

best_d = depths[np.argmax(val_aucs)]
print(f"Best max_depth = {best_d}  (CV AUC = {max(val_aucs):.4f})")

plt.figure(figsize=(8, 4))
plt.plot(depths, train_aucs, 'o-', color='#3266ad', label='Train AUC')
plt.plot(depths, val_aucs,   'o-', color='#e05a4a', label='CV Val AUC')
plt.axvline(best_d, linestyle='--', color='gray', lw=1, label=f'Best depth={best_d}')
plt.xlabel('Max depth'); plt.ylabel('AUC-ROC')
plt.title('Decision Tree — Depth vs. Performance'); plt.legend()
plt.tight_layout(); plt.show()